In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
import joblib

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.20.0


In [2]:
nome_arquivo = '../datasets/dataset2.csv' 
df = pd.read_csv(nome_arquivo, sep=',') 

df.columns = df.columns.str.replace('#', '').str.strip().str.lower()
df.rename(columns={'heart_rate': 'bpm', 'activityid': 'target'}, inplace=True)

# Remove o ruído
df = df[df['target'] != 'transient activities'].copy()

# Rótulo
le = LabelEncoder()
df['target'] = le.fit_transform(df['target']) 

# 0 = Repouso (lying, sitting, standing associados aos códigos 0, 1, 2)
# 1 = Atividade/Anormal (o restante)
df['target'] = df['target'].apply(lambda x: 0 if x <= 2 else 1)
df.dropna(inplace=True)

print(f"Dados carregados e limpos. Shape atual: {df.shape}")

Dados carregados e limpos. Shape atual: (1936481, 33)


In [3]:
# Para prever problemas de saúde
window_size = 5

# Criação de janelas deslizantes
df['rolling_mean'] = df['bpm'].rolling(window=window_size).mean()
df['rolling_std'] = df['bpm'].rolling(window=window_size).std()
df['diff'] = df['bpm'].diff()

df.dropna(inplace=True)

# Definição das variáveis independentes (X) e dependente (y)
features = ['bpm', 'rolling_mean', 'rolling_std', 'diff']
X = df[features].values
y = df['target'].values

print(f"Features extraídas. Total de amostras prontas: {X.shape[0]}")

Features extraídas. Total de amostras prontas: 1936477


In [4]:
# Separação Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dados escalonados com sucesso.")

Dados escalonados com sucesso.


In [5]:
# Construção do modelo Sequencial
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)), 
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid") # Saída probabilística (0 a 1)
])

# Compilação do modelo
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.Recall(name='recall')]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,433 (9.50 KB)

 Trainable params: 2,433 (9.50 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
class_weight_dict = {0: 3.0, 1: 1.0}

# O histórico guarda as métricas por época para gráficos depois
history = model.fit(
    X_train_scaled, 
    y_train, 
    epochs=15, 
    batch_size=32,
    validation_split=0.2, 
    class_weight=class_weight_dict,
    verbose=1
)

Epoch 1/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 54s 2ms/step - accuracy: 0.7943 - loss: 0.5979 - recall: 0.7654 - val_accuracy: 0.7873 - val_loss: 0.4165 - val_recall: 0.7477
Epoch 2/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 47s 1ms/step - accuracy: 0.7966 - loss: 0.5900 - recall: 0.7696 - val_accuracy: 0.7851 - val_loss: 0.4188 - val_recall: 0.7435
Epoch 3/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 67s 2ms/step - accuracy: 0.7975 - loss: 0.5889 - recall: 0.7713 - val_accuracy: 0.7927 - val_loss: 0.4103 - val_recall: 0.7605
Epoch 4/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 74s 2ms/step - accuracy: 0.7984 - loss: 0.5884 - recall: 0.7726 - val_accuracy: 0.7875 - val_loss: 0.4157 - val_recall: 0.7482
Epoch 5/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 63s 2ms/step - accuracy: 0.7976 - loss: 0.5887 - recall: 0.7713 - val_accuracy: 0.8118 - val_loss: 0.4030 - val_recall: 0.7980
Epoch 6/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 59s 2ms/step - accuracy: 0.7968 - loss: 0.5893 - recall: 0.7694 - val_accuracy: 0.7832 - val_loss: 0.

In [7]:
# Realiza as previsões no conjunto de teste
probs_ativ = model.predict(X_test_scaled)

# Converte P(Atividade) para P(Repouso)
probs_repouso = 1.0 - probs_ativ.flatten()

# Limiar de segurança para a área da saúde
THRESHOLD = 0.8

# Se P(Repouso) >= 0.8 classifica como 0, senão 1
preds_custom = np.where(probs_repouso >= THRESHOLD, 0, 1)

print("\n--- Resultados Finais da Avaliação ---")
print("\nMatriz de Confusão:")
print(confusion_matrix(y_test, preds_custom))

print("\nRelatório de Classificação (Threshold = 0.8):")
print(classification_report(y_test, preds_custom, target_names=['Repouso (0)', 'Atividade/Anormal (1)']))

18155/18155 ━━━━━━━━━━━━━━━━━━━━ 15s 837us/step

--- Resultados Finais da Avaliação ---

Matriz de Confusão:
[[ 79383  61594]
 [ 33608 406359]]

Relatório de Classificação (Threshold = 0.8):
                       precision    recall  f1-score   support

          Repouso (0)       0.70      0.56      0.63    140977
Atividade/Anormal (1)       0.87      0.92      0.90    439967

             accuracy                           0.84    580944
            macro avg       0.79      0.74      0.76    580944
         weighted avg       0.83      0.84      0.83    580944



In [8]:
print("\n" + "="*50)
print(" INICIANDO AVALIAÇÃO DOS BASELINES DE COMPARAÇÃO ")
print("="*50)

# 1. Baseline 'Cego' (Dummy Classifier)
# Estratégia: Chuta sempre a classe mais frequente do banco de dados
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train_scaled, y_train)
dummy_preds = dummy_clf.predict(X_test_scaled)

print("\n--- Baseline 1: Dummy Classifier (Estratégia: Mais Frequente) ---")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, dummy_preds))
print("\nRelatório de Classificação:")
# zero_division=0 evita warnings caso ele nunca preveja a classe 1
print(classification_report(y_test, dummy_preds, target_names=['Repouso (0)', 'Atividade/Anormal (1)'], zero_division=0)) 


# 2. Baseline Clássico (Regressão Logística)
# Estratégia: Algoritmo linear tradicional usando os mesmos pesos do Keras
log_reg = LogisticRegression(class_weight={0: 3.0, 1: 1.0}, random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
log_preds = log_reg.predict(X_test_scaled)

print("\n--- Baseline 2: Regressão Logística ---")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, log_preds))
print("\nRelatório de Classificação:")
print(classification_report(y_test, log_preds, target_names=['Repouso (0)', 'Atividade/Anormal (1)']))


 INICIANDO AVALIAÇÃO DOS BASELINES DE COMPARAÇÃO 

--- Baseline 1: Dummy Classifier (Estratégia: Mais Frequente) ---
Matriz de Confusão:
[[     0 140977]
 [     0 439967]]

Relatório de Classificação:
                       precision    recall  f1-score   support

          Repouso (0)       0.00      0.00      0.00    140977
Atividade/Anormal (1)       0.76      1.00      0.86    439967

             accuracy                           0.76    580944
            macro avg       0.38      0.50      0.43    580944
         weighted avg       0.57      0.76      0.65    580944


--- Baseline 2: Regressão Logística ---
Matriz de Confusão:
[[118366  22611]
 [107457 332510]]

Relatório de Classificação:
                       precision    recall  f1-score   support

          Repouso (0)       0.52      0.84      0.65    140977
Atividade/Anormal (1)       0.94      0.76      0.84    439967

             accuracy                           0.78    580944
            macro avg       0.73      

In [9]:
# Salva o modelo treinado
model.save("neuralNetwork/models/clinical_integrated_model.keras")

# Salva o padronizador
joblib.dump(scaler, "neuralNetwork/models/scaler.pkl")

['neuralNetwork/models/scaler.pkl']